# Introduction

# Stage 05 — Inference: chance, increment, and gap magnitude

**Pipeline position:** fifth, and the decisive one. Reads the row-level frame; writes the comparison
grid.

## Two questions that are constantly confused, kept separate here

**Question 1 — does the agreement cell beat chance?** Answered by an empirical permutation null.
This is the *easy* question, and a study that answers only this one and reports it as if it answered
the second would be wrong in the most flattering direction possible.

**Question 2 — does our model add anything to Sleeper alone?** Sleeper's projection is free and
public. If agreement works only because Sleeper works, our model contributes nothing and has no claim
to a place beside it. Answered by a stratified bootstrap on two conditional comparators.

The notebook computes them in that order and never combines them.

## The comparators, and which one matters

1. **Among *model* calls above the threshold**, compare rows where Sleeper agrees against rows where
   Sleeper does not — both graded against the *model's* direction. Asks: does Sleeper's endorsement
   improve our calls? This is the **weak** direction.
2. **Among *Sleeper* calls above the threshold**, compare rows where our model agrees against rows
   where it does not — both graded against *Sleeper's* direction. Asks: does our endorsement improve
   Sleeper's calls? **This is the direction that decides the study.**

A CI that crosses zero means the data are consistent with no improvement. It is not proof of no
effect, but on a post-hoc study with no pre-registration, failure to demonstrate is where the honest
reading stops.

## Third check — does agreement matter beyond gap size?

Agreement cells are mechanically also the large-gap cells, since eligibility requires both magnitudes
above the threshold. A descriptive logistic separates the two. Note its outcome is **our model's**
call being correct, so like comparator (1) it can only speak to the weak direction.

## Independence is not assumed

Sleeper's projection is public and our model trains on overlapping information about the same
players. These are correlated opinions, and no "two independent signals" reading is available
anywhere in this notebook.

## Inputs and outputs

| Direction | Path |
|---|---|
| in | `artifacts/player_season_results.csv`, `00_shared_pipeline.ipynb` |
| out | `artifacts/incremental_comparisons.csv`, `interim/stage05_logistic.json` |

### Explain — load the shared library

Every stage notebook begins here. It loads `00_shared_pipeline.ipynb` using the repo's convention
(`memory/prefer-ipynb-not-py.md`, mirroring the loader in `betting/predict_totals.ipynb` cell 4):
**json + exec over the library's code cells**, never `%run` (brittle across nbclient / papermill /
VSCode) and never a `.py` module (the repo is notebook-centric by rule).

`RUN_TESTS = False` and `SHARED_VERBOSE = False` are set **before** the exec, so the library's inline
tests are skipped and its configuration banner stays silent — those belong to a standalone run of the
library, not to every consumer.

The cell prints a compact load record: the SHA-256 of the library notebook itself, the count of names
imported, and the pinned parameters. Recording the library's hash means each stage's output states
exactly which version of the shared code produced it — if the library changes, the stages' recorded
hashes diverge and the mismatch is visible rather than silent.

In [1]:
import json as _json
from pathlib import Path as _Path


def _exec_notebook(path, glob):
    """Execute every code cell of a notebook into `glob` (repo convention: json + exec)."""
    with open(path, encoding="utf-8") as _fh:
        _nb = _json.load(_fh)
    for _cell in _nb["cells"]:
        if _cell["cell_type"] == "code":
            exec("".join(_cell["source"]), glob)


RUN_TESTS = False          # skip the library's inline self-tests in a consumer
SHARED_VERBOSE = False     # suppress the library's configuration banner
_SHARED = "00_shared_pipeline.ipynb"
_before = set(globals())
_exec_notebook(_SHARED, globals())
_loaded = sorted(n for n in set(globals()) - _before
                 if not n.startswith("_") and n not in {"RUN_TESTS", "SHARED_VERBOSE"})

print(f"loaded {_SHARED}")
print(f"  library sha256 : {sha256_file(_SHARED)}")
print(f"  names imported : {len(_loaded)}")
print(f"  functions      : {[n for n in _loaded if callable(globals()[n])]}")
print(f"  repo           : {REPO.name}   project: {PROJECT.name}")
print(f"  seasons {TEST_SEASONS} | thresholds {THRESHOLDS} | populations {list(POPULATIONS)}")
print(f"  seed {SEED} | perms {N_PERM:,} | boots {N_BOOT:,}")

loaded 00_shared_pipeline.ipynb
  library sha256 : d3e28a60fab75caf19c5387de293057de842c21c3c088edbc204acd89972b469
  names imported : 48
  functions      : ['Path', 'add_signals', 'boot_index_matrix', 'bootstrap_lift', 'build_ranks', 'canonical_strata', 'correct_vec', 'datetime', 'logistic_design', 'logistic_newton', 'norm', 'perm_sign_matrix', 'permutation_test', 'population_slice', 'sha256_file', 'spearmanr', 'summarise_cell', 'thr_col', 'timezone', 'wilson']
  repo           : JoSchoAnalytics   project: adp_consensus_agreement_2026-08-02
  seasons [2021, 2022, 2023, 2024, 2025] | thresholds [0.0, 5.0, 7.5, 10.0] | populations ['all_adp', 'drafted_top180']
  seed 20260802 | perms 10,000 | boots 10,000


### Interpretation — library loaded, this stage is anchored to it

The load record confirms the shared library executed cleanly and lists the names now in scope,
including the analysis functions this stage calls. The pinned parameters match the study's
declaration — seasons 2021–2025, thresholds `[0, 5, 7.5, 10]`, both populations, seed 20260802 — so
this notebook cannot silently disagree with its siblings about what a rank is or how a hit rate is
scored.

The **library SHA-256 is printed and recorded**. Every stage prints the same digest, which is what
makes "all seven stages ran against the same library" a checkable claim rather than an assumption;
stage 07 re-hashes the library and compares.

`RUN_TESTS=False` means the library's self-tests did not run here — they belong to a standalone run
of `00_shared_pipeline.ipynb`, which is the gate for this pipeline being trustworthy at all.

**This stage reads:** `artifacts/player_season_results.csv` and the shared library
**and writes:** `artifacts/incremental_comparisons.csv` and `interim/stage05_logistic.json`

### Explain — the empirical null: does the agreement cell beat chance?

A 66% or 85% hit rate means nothing until we know what chance would give. The naive reference is 50%,
but that is an assumption: ties, direction imbalance, and the particular mix of season-position cells
could all shift the true chance level away from a half. So it is measured rather than assumed.

**The permutation.** `actual_gap` is shuffled **within each `(season, position)` cell**, 10,000 times.
Everything else is held fixed — which rows are in the agreement cell, each row's predicted direction,
the thresholds, the cell sizes. Only the pairing between a player and his realised outcome is
destroyed.

Shuffling within season-position rather than globally is the point: it preserves each cell's own
distribution of `actual_gap`, including its spread, skew and tie mass. The null then answers "how
often would *these specific calls* be right if outcomes were assigned at random among *these specific
players*?" — not the weaker "how often does a coin land heads?"

**Implementation.** One `(n_perm x N)` int8 sign matrix per panel, reused across all four thresholds,
so every threshold is tested against the same null draws. Seeded, so the p-values are reproducible.

**Reported.** Observed rate, null mean, null 95th percentile, and a one-sided p-value
`(#{null >= observed} + 1) / (n_perm + 1)`. The `+1` makes it conservative and bounds it away from
zero; with 10,000 draws the floor is ~0.0001, and a p at that floor means "not one of 10,000
relabelings matched", not "p = 0".

**Scope.** This answers *only* whether the cell beats chance. It says nothing about whether our model
adds anything to Sleeper — that is the next cell, and conflating the two is the central risk of this
study.

In [2]:
PLAYER_RESULTS = pd.read_csv(ARTIFACTS / "player_season_results.csv")
SIGNALS = PLAYER_RESULTS.rename(columns={"position": "pos", "model_pred": "pred",
                                         "actual_half_ppr": "y", "model_family": "model"})
print(f"loaded {len(SIGNALS):,} row-level records\n")

PERM_ROWS, PANEL_CACHE = [], {}
for pop_name in POPULATIONS:
    for uni in ("A", "B"):
        for panel in POOLED_PANELS:
            d = (SIGNALS[(SIGNALS.population == pop_name) & (SIGNALS.universe == uni)
                         & SIGNALS.complete & SIGNALS.season.isin(PANELS[panel])]
                 .reset_index(drop=True))
            sm = perm_sign_matrix(d)
            PANEL_CACHE[(pop_name, uni, panel)] = d
            for t in THRESHOLDS:
                PERM_ROWS.append({"population": pop_name, "universe": uni, "panel": panel,
                                  "threshold": t, "panel_complete_n": len(d),
                                  **permutation_test(d, sm, d[thr_col(t)], "consensus_score")})
PERM = pd.DataFrame(PERM_ROWS)

print(f"PERMUTATION NULL — {N_PERM:,} within-(season, position) shuffles of actual_gap, seed={SEED}")
print("=" * 112)
for pop_name in POPULATIONS:
    print(f"\n-- {pop_name}, universe A --")
    print(PERM[(PERM.population == pop_name) & (PERM.universe == "A")]
          .sort_values(["panel", "threshold"])
          [["panel", "threshold", "n", "observed", "null_mean", "null_p95", "p_value"]]
          .round(4).to_string(index=False))

print("\n" + "-" * 112)
print(f"null mean across ALL {len(PERM)} cells : min {PERM.null_mean.min():.4f}  max {PERM.null_mean.max():.4f}")
print(f"null 95th percentile                  : min {PERM.null_p95.min():.4f}  max {PERM.null_p95.max():.4f}")
print(f"p-value                               : min {PERM.p_value.min():.6f}  max {PERM.p_value.max():.6f}")
print(f"p-value resolution floor 1/(n+1)      : {1/(N_PERM+1):.6f}")
print(f"cells with observed <= their own null 95th pct : {int((PERM.observed <= PERM.null_p95).sum())} of {len(PERM)}")

loaded 5,315 row-level records



PERMUTATION NULL — 10,000 within-(season, position) shuffles of actual_gap, seed=20260802

-- all_adp, universe A --
           panel  threshold   n  observed  null_mean  null_p95  p_value
pooled_2021_2025        0.0 940    0.7681     0.4936    0.5202   0.0001
pooled_2021_2025        5.0 516    0.8566     0.5001    0.5349   0.0001
pooled_2021_2025        7.5 422    0.8815     0.5022    0.5403   0.0001
pooled_2021_2025       10.0 331    0.9124     0.5036    0.5468   0.0001
pooled_2023_2025        0.0 655    0.7802     0.4979    0.5298   0.0001
pooled_2023_2025        5.0 392    0.8801     0.5024    0.5434   0.0001
pooled_2023_2025        7.5 338    0.9024     0.5041    0.5473   0.0001
pooled_2023_2025       10.0 281    0.9253     0.5043    0.5516   0.0001
pooled_2024_2025        0.0 512    0.7988     0.5002    0.5352   0.0001
pooled_2024_2025        5.0 338    0.8876     0.5032    0.5473   0.0001
pooled_2024_2025        7.5 307    0.9023     0.5045    0.5505   0.0001
pooled_2024_2025   

### Interpretation — the cell beats chance decisively, and the 50% reference is empirically correct

Across all **48 permutation cells** the null mean lands between **0.4831 and 0.5050**. That is the
measured answer to what had been an assumption: after preserving each season-position cell's own
distribution of outcomes, ties and direction imbalance included, chance really is approximately a coin
flip. The informal 50% reference used elsewhere is legitimate — now verified rather than asserted.

Every cell returns **p = 0.0001**, the resolution floor for 10,000 draws: **not one of 10,000 random
relabelings** reached the observed hit rate, in any panel, at any threshold, in either population.
That is not "p is zero"; a larger budget would only push the bound lower.

The strongest single statement is the last line: **0 of 48 cells have an observed rate at or below
their own null 95th percentile.** Even the smallest and most fragile clear their own bar — the drafted
board at t>10 observed 1.0000 against a null p95 of **0.7333** on 15 calls, and the five-season panel
at t>10 observed 0.8919 against 0.6216 on 37. Note how the null p95 widens as cells shrink (0.5176 at
the largest, 0.7333 at the smallest): the test correctly demands more from a small cell.

**What this does not settle, and it is the point of the next cell.** Beating chance is not the question
Joseph asked. Sleeper's projection alone would also beat chance, comfortably. The permutation says the
agreement cell contains real information; it says nothing about *whose* information it is.

### Explain — the comparison that actually decides the study

Sleeper's projection is free and public. The question that matters is whether *our model's agreement*
adds anything to it. This cell isolates that with two comparators, both conditioned on a call already
existing.

1. **Among model calls above the threshold** (`|model_gap| > t`), compare rows where Sleeper agrees
   against rows where Sleeper does not — both graded against the *model's* direction. Asks: does
   Sleeper's endorsement improve our calls? **The weak direction.**
2. **Among Sleeper calls above the threshold** (`|sleeper_gap| > t`), compare rows where our model
   agrees against rows where it does not — both graded against *Sleeper's* direction. Asks: does our
   endorsement improve Sleeper's calls? **This is the direction that decides the study**, because it
   is the only one that could justify our model earning a place beside a projection that already
   exists.

**Uncertainty.** A stratified bootstrap resamples within `(season, position)` 10,000 times, preserving
panel composition, and reports a percentile 95% CI on the *difference* in hit rates. One index matrix
per panel is reused across thresholds and both comparators. `ci_crosses_zero` is returned as a field
so the verdict-critical boolean lands in the exported CSV rather than being re-derived by a reader.

**How to read a CI that crosses zero.** The data are consistent with no improvement. That is not proof
of no effect, but on a post-hoc study with no pre-registration, failure to demonstrate is where the
honest reading stops.

**Independence is not claimed.** Sleeper is public and our model trains on overlapping information
about the same players; these are correlated opinions.

In [3]:
COMP_ROWS = []
for (pop_name, uni, panel), d in PANEL_CACHE.items():
    bi = boot_index_matrix(d)
    for t in THRESHOLDS:
        both = d[thr_col(t)]
        model_call = (d.model_gap.abs() > t) & (d.model_gap != 0)
        sleeper_call = (d.sleeper_gap.abs() > t) & (d.sleeper_gap != 0)
        v_model = bootstrap_lift(d, bi, model_call & both, model_call & ~both, "model_gap", "model_gap")
        v_sleep = bootstrap_lift(d, bi, sleeper_call & both, sleeper_call & ~both,
                                 "sleeper_gap", "sleeper_gap")
        COMP_ROWS.append({"population": pop_name, "universe": uni, "panel": panel, "threshold": t,
                          "panel_complete_n": len(d),
                          **{f"vs_model_alone_{k}": v for k, v in v_model.items()},
                          **{f"vs_sleeper_alone_{k}": v for k, v in v_sleep.items()}})
COMPARISONS = pd.DataFrame(COMP_ROWS)

_SC = ["threshold", "vs_sleeper_alone_hr_agree", "vs_sleeper_alone_hr_no_agree",
       "vs_sleeper_alone_n_agree", "vs_sleeper_alone_n_no_agree", "vs_sleeper_alone_lift",
       "vs_sleeper_alone_ci_lo", "vs_sleeper_alone_ci_hi", "vs_sleeper_alone_ci_crosses_zero"]
_MC = ["threshold", "vs_model_alone_hr_agree", "vs_model_alone_hr_no_agree",
       "vs_model_alone_lift", "vs_model_alone_ci_lo", "vs_model_alone_ci_hi",
       "vs_model_alone_ci_crosses_zero"]

print("INCREMENTAL COMPARISONS — stratified bootstrap, "
      f"{N_BOOT:,} resamples, seed={SEED+1}, universe A")
for pop_name in POPULATIONS:
    for panel in POOLED_PANELS:
        t = COMPARISONS[(COMPARISONS.population == pop_name) & (COMPARISONS.universe == "A")
                        & (COMPARISONS.panel == panel)].sort_values("threshold")
        print("\n" + "=" * 118)
        print(f"{pop_name}  |  {panel}")
        print("=" * 118)
        print("  (2) among SLEEPER calls: does OUR agreement help?   <-- THE DECISIVE DIRECTION")
        print(t[_SC].round(4).to_string(index=False))
        print("\n  (1) among MODEL calls: does Sleeper's agreement help ours?   [weak direction]")
        print(t[_MC].round(4).to_string(index=False))

print("\n" + "=" * 118)
print("VERDICT SCAN — Sleeper-side lift, universe A: does any CI exclude zero?")
print("=" * 118)
for _, r in COMPARISONS[COMPARISONS.universe == "A"].sort_values(
        ["population", "panel", "threshold"]).iterrows():
    v = "CROSSES ZERO -> not demonstrated" if r.vs_sleeper_alone_ci_crosses_zero else "excludes zero"
    print(f"  {r.population:15s} {r.panel:17s} t>{r.threshold:<4g} "
          f"lift {r.vs_sleeper_alone_lift:+.4f} "
          f"[{r.vs_sleeper_alone_ci_lo:+.4f}, {r.vs_sleeper_alone_ci_hi:+.4f}]  {v}")

COMPARISONS_OUT = COMPARISONS.merge(
    PERM.rename(columns={c: f"perm_{c}" for c in
                         ["n", "observed", "null_mean", "null_p95", "p_value"]}),
    on=["population", "universe", "panel", "threshold", "panel_complete_n"], how="left")
COMPARISONS_OUT.to_csv(ARTIFACTS / "incremental_comparisons.csv", index=False)
print(f"\nwrote artifacts/incremental_comparisons.csv — {len(COMPARISONS_OUT)} rows "
      f"(bootstrap lifts joined to permutation nulls)")

INCREMENTAL COMPARISONS — stratified bootstrap, 10,000 resamples, seed=20260803, universe A

all_adp  |  pooled_2024_2025
  (2) among SLEEPER calls: does OUR agreement help?   <-- THE DECISIVE DIRECTION
 threshold  vs_sleeper_alone_hr_agree  vs_sleeper_alone_hr_no_agree  vs_sleeper_alone_n_agree  vs_sleeper_alone_n_no_agree  vs_sleeper_alone_lift  vs_sleeper_alone_ci_lo  vs_sleeper_alone_ci_hi  vs_sleeper_alone_ci_crosses_zero
       0.0                     0.7988                        0.5831                       512                          307                 0.2158                  0.1510                  0.2807                             False
       5.0                     0.8876                        0.7062                       338                          194                 0.1814                  0.1103                  0.2533                             False
       7.5                     0.9023                        0.7423                       307                    

### Interpretation — the decisive comparison does not support an incremental claim

**Comparator (1), the weak direction, is unambiguous.** Among our model's own calls, Sleeper's
agreement lifts the hit rate by **+0.21 to +0.48** on the drafted board and +0.34 to +0.42 on the full
population, with every interval clear of zero in all 24 cells. Sleeper's endorsement makes our calls
much better. Given that the shipped models beat Sleeper at no position (RB ρ +0.689, WR +0.736, TE
+0.734, QB +0.695, all below Sleeper's), that is close to expected — conditioning our calls on a
better forecaster's agreement should help.

**Comparator (2) is the one that could justify the model's place beside Sleeper, and on the drafted
board it fails on the longest panel:**

| panel | t>5 | t>7.5 | t>10 |
|---|---|---|---|
| **2021–2025 (five seasons)** | **+0.068 [−0.028, +0.162]** | **+0.073 [−0.028, +0.172]** | **+0.035 [−0.095, +0.156]** |
| 2023–2025 | +0.194 [+0.056, +0.326] | +0.183 [+0.044, +0.317] | +0.216 [+0.091, +0.351] |
| 2024–2025 | +0.153 [−0.004, +0.311] | +0.181 [+0.026, +0.333] | +0.269 [+0.107, +0.450] |

**On the full five-season drafted panel every interval above t>0 crosses zero.** The shorter panels
look positive, which at first reads as a recent-years effect — but they are nested subsets of the
panel that fails, they hold a third to a half as many calls, and stage 04 showed the per-season
drafted rate *declining* into 2025. The reading that survives all of it: **on the drafted board, this
study does not demonstrate that our model adds value beyond Sleeper alone.** The 2024–25 t>10 lift of
+0.269 rests on 15 agreement calls against 26 non-agreement calls — not a foundation for a claim.

The mechanism is visible in the `hr_no_agree` column: Sleeper's *unaided* drafted-board calls already
hit **77.0%, 81.3% and 85.7%** at the three thresholds over five seasons. The bar our model must clear
rises with the threshold, which is exactly why the lift shrinks to +0.035 at t>10 rather than growing.

**On the full population comparator (2) does clear zero everywhere** (+0.099 to +0.216, all 12 cells).
But stage 03 established that population is 86–92% undrafted at these thresholds, so what it shows is
that our model helps Sleeper order the noise floor. That is not a draft-board claim.

### Explain — does agreement matter beyond gap size?

The agreement cells are also, mechanically, the large-gap cells: eligibility requires both magnitudes
above the threshold. So part of the agreement effect might just be "big disagreements are more often
right". This cell separates the two.

**Model.** Logistic regression fitted by the shared library's Newton–Raphson routine:

```
P(model's directional call is correct) ~ agree + |model_gap| + |sleeper_gap| + position + season
```

**Read the outcome direction carefully.** The outcome is whether **our model's** call was right, and
`agree` is whether Sleeper pointed the same way. A positive `agree` coefficient therefore says
*Sleeper's agreement improves our calls* — the same weak direction as comparator (1). It **cannot**
say our model improves Sleeper, because that outcome does not appear in the fit. The decisive
evidence remains the Sleeper-side bootstrap above.

**Population.** Rows where both projections express a nonzero disagreement and the actual outcome is
not an exact tie; 2024–2025; universe B, so every row is complete by construction. Fitted separately
for both populations.

**Stability reporting.** Convergence, iteration count, McFadden pseudo-R² and an explicit separation
flag are all reported, so an unstable fit is labelled rather than presented as a clean coefficient
table.

**Status.** Descriptive. Nothing is selected on this model and no product uses it.

In [4]:
LOGIT = {}
print("DESCRIPTIVE LOGISTIC — outcome = OUR MODEL's directional call is correct")
print("population: universe B, pooled 2024-2025, both gaps nonzero, actual_gap nonzero")
for pop_name in POPULATIONS:
    d = SIGNALS[(SIGNALS.population == pop_name) & (SIGNALS.universe == "B") & SIGNALS.complete
                & SIGNALS.season.isin([2024, 2025])]
    d = d[(d.model_gap != 0) & (d.sleeper_gap != 0) & (d.actual_gap != 0)]
    y = (np.sign(d.actual_gap) == np.sign(d.model_gap)).astype(float).to_numpy()
    X, names = logistic_design(d)
    fit = logistic_newton(X, y, names)
    fit["baseline_correct_rate"] = float(y.mean())
    fit["agree_share"] = float(d.agree_dir.mean())
    LOGIT[pop_name] = fit
    print("\n" + "=" * 96)
    print(f"{pop_name}: n={fit['n']}  converged={fit['converged']} ({fit['iterations']} iters)  "
          f"pseudo-R2={fit['pseudo_r2']:.4f}  unstable={fit['unstable_or_separated']}")
    print(f"  baseline correct rate {fit['baseline_correct_rate']:.4f} | "
          f"share with Sleeper agreeing {fit['agree_share']:.4f}")
    print("=" * 96)
    tab = pd.DataFrame({"term": names, "coef": fit["coef"], "se": fit["se"],
                        "z": fit["z"], "p": fit["p"]})
    tab["odds_ratio"] = np.exp(tab.coef)
    print(tab.round(4).to_string(index=False))

(INTERIM / "stage05_logistic.json").write_text(json.dumps(LOGIT, indent=2), encoding="utf-8")
print("\nwrote interim/stage05_logistic.json")
print("REMINDER: 'agree' measures Sleeper improving OUR calls. It cannot show the reverse.")

DESCRIPTIVE LOGISTIC — outcome = OUR MODEL's directional call is correct
population: universe B, pooled 2024-2025, both gaps nonzero, actual_gap nonzero

all_adp: n=783  converged=True (6 iters)  pseudo-R2=0.1343  unstable=False
  baseline correct rate 0.6501 | share with Sleeper agreeing 0.6424
           term    coef     se       z      p  odds_ratio
      intercept -0.4875 0.2742 -1.7780 0.0754      0.6141
          agree  1.4323 0.1708  8.3850 0.0000      4.1885
  abs_model_gap  0.0251 0.0069  3.6382 0.0003      1.0254
abs_sleeper_gap  0.0066 0.0070  0.9361 0.3492      1.0066
        pos[RB] -0.2735 0.2865 -0.9545 0.3398      0.7607
        pos[TE] -0.2031 0.3077 -0.6600 0.5093      0.8162
        pos[WR] -0.5010 0.2846 -1.7602 0.0784      0.6059
   season[2025]  0.1185 0.1657  0.7150 0.4746      1.1258

drafted_top180: n=289  converged=True (5 iters)  pseudo-R2=0.0736  unstable=False
  baseline correct rate 0.5571 | share with Sleeper agreeing 0.5363
           term    coef     se

### Interpretation — agreement carries information beyond gap size, in the weak direction

Both fits converged cleanly (6 and 5 Newton iterations) with `unstable_or_separated = False`, so the
coefficients are readable.

`agree` is strongly positive in both: **+1.432** (SE 0.171, p < 1e-15, odds ratio **4.19**) on the
full population and **+1.128** (SE 0.253, p ≈ 8e-6, odds ratio **3.09**) on the drafted board. Because
`|model_gap|` and `|sleeper_gap|` are in the same model, this is **not** explained by agreement cells
being the big-gap cells. Tripling the odds that our model's call is correct is a real effect that
survives controlling for magnitude — which is exactly what this cell exists to test.

Magnitude itself contributes much less than one might assume. `abs_model_gap` is positive but tiny
(+0.025 and +0.040 per rank spot, the drafted-board coefficient only marginally significant at
p = 0.041), and **`abs_sleeper_gap` is insignificant in both fits** (p = 0.35 and p = 0.60), even
negative on the drafted board. So the informative content is *that Sleeper agrees*, not *how emphatic
Sleeper is* — an independent argument for the minimum-gap consensus score, since the weaker gap's
exact size is not carrying much.

Pseudo-R² is 0.134 and 0.074. Low, and appropriately so: whether a specific player beats his ADP is
mostly unpredictable, and a model claiming otherwise on 289 drafted rows would be the suspicious
result.

**The direction limitation is the whole caveat and must not be lost.** The outcome is *our model's*
call being correct and `agree` is *Sleeper agreeing with us*. This says Sleeper improves us — the same
weak direction as comparator (1). **It cannot show that we improve Sleeper.** The decisive evidence
remains the Sleeper-side bootstrap, and it did not clear zero on the five-season drafted panel.

# Conclusion and Next Steps

## What this stage established

**Question 1 — chance: answered decisively yes.** Across all **48 permutation cells** the null mean
sits between **0.4831 and 0.5050**, so the informal 50% reference is empirically correct rather than
assumed. Every cell returns **p = 0.0001**, the floor for 10,000 draws — not one random relabeling
reached the observed rate. **0 of 48** cells fail to clear their own null 95th percentile, including
the smallest: drafted board at t>10, observed 1.0000 against a null p95 of 0.7333 on 15 calls.

**Question 2 — increment: not established on the drafted board.** Among Sleeper's own calls, adding
our model's agreement gives, over the full **2021–2025** panel:

| threshold | lift | 95% CI | n (agree / no-agree) |
|---|---|---|---|
| >5 | +0.068 | **[−0.028, +0.162]** | 105 / 161 |
| >7.5 | +0.073 | **[−0.028, +0.172]** | 70 / 128 |
| >10 | +0.035 | **[−0.095, +0.156]** | 37 / 77 |

**Every interval crosses zero.** The 2023–2025 and 2024–2025 panels do show lifts clear of zero at the
higher thresholds, but they are nested subsets of the panel that fails, they hold a third to a half as
many calls, and stage 04 showed the per-season rate declining into 2025.

**The weak direction is strong**, as expected: among *our* calls, Sleeper's agreement lifts the hit
rate by **+0.21 to +0.48** on the drafted board, every interval clear of zero in all 24 cells. Given
the shipped models beat Sleeper at no position, conditioning our calls on a better forecaster's
agreement should help — and it does.

**The mechanism is visible in the comparator.** Sleeper's *unaided* drafted-board calls already hit
**77.0% / 81.3% / 85.7%** at the three thresholds over five seasons. The bar our model must clear
rises with the threshold, which is exactly why the lift shrinks to +0.035 at t>10 rather than growing.

**Gap magnitude does not explain agreement.** `agree` is +1.128 (SE 0.253, p ≈ 8e-6, odds ratio
**3.09**) on the drafted board with both gap magnitudes in the model. But `abs_sleeper_gap` is
insignificant in both fits (p = 0.60 and 0.35), so what carries information is *that* Sleeper agrees,
not *how emphatically* — an independent argument for the weaker-gap consensus score. Both fits
converged with no separation flag; pseudo-R² is 0.074 and 0.134, appropriately low.

## The verdict this stage supports

**The agreement cell picks the right side of ADP well above chance. On the drafted board, this study
cannot show that our model adds anything to Sleeper's projection on its own.**

## Next step

Run **`06_freshness_and_player_audit.ipynb`** — there is a named confound in the prereg that has not
yet been confronted, and the individual calls have not yet been looked at.